# Notebook 03: Machine Learning Model Benchmarking & Explainable AI (SHAP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadrahman444/scientific-research-and-methodologies/blob/main/colab_notebooks/03_Machine_Learning_Crop_Yield_Prediction_Colab.ipynb)

**Research Topic:** A Data-Driven Analysis of 15-Year Weather Variability and Its Impact on Agricultural Crop Yield in Bangladesh  
**Methodological Framework:** Two-Tier Quantitative Design (Option A)  
**Cloud Execution:** Google Colab (Zero local GPU overhead)  

---
## 1. Machine Learning Modeling Framework
Trains and benchmarks Ridge Regression, Random Forest, XGBoost, LightGBM, and CatBoost on both the Sub-National multi-crop dataset ($N=768$) and National FAOSTAT series, followed by TreeSHAP interpretability.


In [ ]:
# Step 1: Install Cloud Dependencies in Colab
!pip install -q xgboost catboost lightgbm shap

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import shap

print('Colab ML environment verified.')


In [ ]:
# Step 2: Load Both Datasets
INTEGRATED_LOCAL = '../data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
INTEGRATED_URL = f'{BASE_RAW}/data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
REGIONAL_LOCAL = '../data/processed/bangladesh_weather_crop_15yr_processed.csv'
REGIONAL_URL = f'{BASE_RAW}/data/processed/bangladesh_weather_crop_15yr_processed.csv'

df_nat = pd.read_csv(INTEGRATED_LOCAL) if os.path.exists(INTEGRATED_LOCAL) else pd.read_csv(INTEGRATED_URL)
df_reg = pd.read_csv(REGIONAL_LOCAL) if os.path.exists(REGIONAL_LOCAL) else pd.read_csv(REGIONAL_URL)
print(f'Ready to train on {len(df_reg)} regional multi-crop records.')


In [ ]:
# Step 3: Feature Engineering and Out-of-Sample Train/Test Partitioning
# Sub-National Tier: 768 samples across 8 divisions & 6 crops
df_encoded = pd.get_dummies(df_reg, columns=['crop', 'division'], drop_first=True)
drop_cols = ['year', 'crop_season', 'yield_mt_ha', 'temp_anomaly', 'rainfall_anomaly_pct']
feat_cols = [c for c in df_encoded.columns if c not in drop_cols]

X_reg = df_encoded[feat_cols]
y_reg = df_encoded['yield_mt_ha']

# Temporal Split: 2009-2021 (Train: 13 years), 2022-2024 (Blind Test: 3 years)
train_idx = df_encoded['year'] <= 2021
test_idx = df_encoded['year'] > 2021

X_train, y_train = X_reg[train_idx], y_reg[train_idx]
X_test, y_test = X_reg[test_idx], y_reg[test_idx]

print(f'Training Samples: {len(X_train)} | Test Samples: {len(X_test)} | Features: {X_train.shape[1]}')


In [ ]:
# Step 4: Model Benchmarking on Sub-National Dataset
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=180, learning_rate=0.06, max_depth=5, random_state=42),
    'LightGBM': lgb.LGBMRegressor(n_estimators=180, learning_rate=0.06, num_leaves=31, random_state=42, verbose=-1),
    'CatBoost': cb.CatBoostRegressor(iterations=200, learning_rate=0.06, depth=5, random_seed=42, verbose=0)
}

benchmark = []
preds = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)
    preds[name] = y_hat
    
    rmse = np.sqrt(mean_squared_error(y_test, y_hat))
    mae = mean_absolute_error(y_test, y_hat)
    r2 = r2_score(y_test, y_hat)
    mape = np.mean(np.abs((y_test - y_hat) / y_test)) * 100
    
    benchmark.append({
        'Model Architecture': name,
        'R2 Score': round(r2, 4),
        'RMSE (MT/ha)': round(rmse, 4),
        'MAE (MT/ha)': round(mae, 4),
        'MAPE (%)': round(mape, 2)
    })

bench_df = pd.DataFrame(benchmark).sort_values(by='R2 Score', ascending=False)
print('=== MODEL BENCHMARK RESULTS (OUT-OF-SAMPLE TEST SET 2022-2024) ===')
display(bench_df)


In [ ]:
# Step 5: Visualizing Predictions vs Ground Truth
best_m = bench_df.iloc[0]['Model Architecture']
plt.figure(figsize=(9, 7))
plt.scatter(y_test, preds[best_m], color='#1f77b4', alpha=0.7, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2.5, label='1:1 Perfect Agreement')
plt.title(f'Ground Truth vs Predicted Yields ({best_m})', fontsize=13, fontweight='bold')
plt.xlabel('Ground Truth Yield (MT/ha)')
plt.ylabel('Model Prediction (MT/ha)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Step 6: TreeSHAP Feature Attribution (Unpacking the Black Box)
explainer = shap.TreeExplainer(models['XGBoost'])
shap_values = explainer.shap_values(X_test)

# Summary Bar Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('Global Feature Importance Ranking via Mean |SHAP Value|', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Beeswarm Plot (Direction of Impact)
plt.figure(figsize=(11, 7))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Beeswarm Plot: Climatic Drivers and Direction of Yield Impact', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
